# **Projeto: Previsão de Intenção de Compra de Clientes em Loja Web**

**Descrição do Projeto**

Neste projeto, nosso objetivo é criar um sistema inteligente para antecipar a intenção de compra dos clientes em um site de e-commerce. Imagine poder prever quais clientes têm maior probabilidade de realizar compras online, com base em suas características e comportamentos passados. Essa capacidade de prever a intenção de compra não só aprimorará a experiência do cliente, mas também permitirá que a empresa direcione seus esforços de marketing de forma mais eficaz.

**Objetivo**

Queremos desenvolver um modelo preditivo capaz de analisar os padrões de comportamento dos clientes e identificar sinais que indicam a propensão deles para realizar compras no site da empresa. Para isso, vamos usar uma base de dados que contém informações detalhadas sobre os clientes, incluindo:

Dados demográficos (idade, renda, etc.)

Informações sobre compras anteriores



# Base de dados:

Year_Birth: Ano de nascimento do cliente.

Education: Nível de escolaridade do cliente.

Marital_Status: Estado civil do cliente.

Income: Renda anual da família do cliente.

Kidhome: Número de crianças na casa do cliente.

Recency: Número de dias desde a última compra do cliente.

Complain: 1 se o cliente reclamou nos últimos 2 anos, 0 caso contrário.

MntWines: Valor gasto em vinhos nos últimos 2 anos.

MntFruits: Valor gasto em frutas nos últimos 2 anos.

MntMeatProducts: Valor gasto em carnes nos últimos 2 anos.

MntFishProducts: Valor gasto em peixes nos últimos 2 anos.

MntSweetProducts: Valor gasto em doces nos últimos 2 anos.

MntGoldProds: Valor gasto em produtos de ouro nos últimos 2 anos.

NumDealsPurchases: Número de compras feitas com desconto

NumStorePurchases: Número de compras feitas diretamente nas lojas.

NumWebVisitsMonth: Número de visitas ao site da empresa no último mês.






**WebPurchases: 0 se não realizou compras pelo site da empresa e 1 se realizou.**

In [63]:
import pandas as pd
import numpy as np
import plotly.express as px
from plotly.subplots import make_subplots
import plotly.graph_objects as go
import math
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.model_selection import KFold, cross_validate, cross_val_predict, cross_val_score
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import confusion_matrix, classification_report, average_precision_score, roc_auc_score
from sklearn.decomposition import PCA

# ETAPA 1:

**Preparação dos Dados**

**Exploração e Limpeza:** Analisar e limpar os dados para garantir que estejam prontos para a modelagem.

**Análise:** Construa uma storytelling com gráficos, analisando e retirando insights das informações.

In [64]:
base = pd.read_csv('marketing_campaign.csv', delimiter=';')
base.head(10)

,Year_Birth,Education,Marital_Status,Income,Kidhome,Recency,MntWines,MntFruits,MntMeatProducts,MntFishProducts,MntSweetProducts,MntGoldProds,NumStorePurchases,NumWebVisitsMonth,Complain,WebPurchases
0,1957,Graduation,Single,58138.0,0,58,635,88,546,172,88,88,4,7,0,1
1,1954,Graduation,Single,46344.0,1,38,11,1,6,2,1,6,2,5,0,0
2,1965,Graduation,Together,71613.0,0,26,426,49,127,111,21,42,10,4,0,1
3,1984,Graduation,Together,26646.0,1,26,11,4,20,10,3,5,4,6,0,0
4,1981,PhD,Married,58293.0,1,94,173,43,118,46,27,15,6,5,0,1
5,1967,Master,Together,62513.0,0,16,520,42,98,0,42,14,10,6,0,1
6,1971,Graduation,Divorced,55635.0,0,34,235,65,164,50,49,27,7,6,0,1
7,1985,PhD,Married,33454.0,1,32,76,10,56,3,1,23,4,8,0,1
8,1974,PhD,Together,30351.0,1,19,14,0,24,3,3,2,2,9,0,0
9,1950,PhD,Together,5648.0,1,68,28,0,6,1,1,13,0,20,0,0


Após a importação da base de dados, é adequado realizar uma checagem inicial de integridade para identificar inconsistências básicas que podem comprometer as análises posteriores. Nessa etapa, verifica-se se os tipos das variáveis estão corretos, se existem valores ausentes em proporções relevantes, se há registros duplicados e se aparecem valores inválidos ou fora de faixas esperadas, como idade negativa, renda igual a zero ou categorias inconsistentes.

In [65]:
base.describe().T

,count,mean,std,min,25%,50%,75%,max
Year_Birth,2240.0,1968.805804,11.984069,1893.0,1959.00,1970.0,1977.00,1996.0
Income,2216.0,52247.251354,25173.076661,1730.0,35303.00,51381.5,68522.00,666666.0
Kidhome,2240.0,0.444196,0.538398,0.0,0.00,0.0,1.00,2.0
Recency,2240.0,49.109375,28.962453,0.0,24.00,49.0,74.00,99.0
MntWines,2240.0,303.935714,336.597393,0.0,23.75,173.5,504.25,1493.0
MntFruits,2240.0,26.302232,39.773434,0.0,1.00,8.0,33.00,199.0
MntMeatProducts,2240.0,166.950000,225.715373,0.0,16.00,67.0,232.00,1725.0
MntFishProducts,2240.0,37.525446,54.628979,0.0,3.00,12.0,50.00,259.0
MntSweetProducts,2240.0,27.062946,41.280498,0.0,1.00,8.0,33.00,263.0
MntGoldProds,2240.0,44.021875,52.167439,0.0,9.00,24.0,56.00,362.0


In [66]:
base.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2240 entries, 0 to 2239
Data columns (total 16 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   Year_Birth         2240 non-null   int64  
 1   Education          2240 non-null   object 
 2   Marital_Status     2240 non-null   object 
 3   Income             2216 non-null   float64
 4   Kidhome            2240 non-null   int64  
 5   Recency            2240 non-null   int64  
 6   MntWines           2240 non-null   int64  
 7   MntFruits          2240 non-null   int64  
 8   MntMeatProducts    2240 non-null   int64  
 9   MntFishProducts    2240 non-null   int64  
 10  MntSweetProducts   2240 non-null   int64  
 11  MntGoldProds       2240 non-null   int64  
 12  NumStorePurchases  2240 non-null   int64  
 13  NumWebVisitsMonth  2240 non-null   int64  
 14  Complain           2240 non-null   int64  
 15  WebPurchases       2240 non-null   int64  
dtypes: float64(1), int64(13)

Ao observar a descrição da base de dados, identifica-se a necessidade de padronizar os tipos das variáveis para formatos mais adequados, garantindo consistência e evitando problemas nas etapas de exploração e modelagem. Variáveis categóricas como Education e Marital_Status devem ser tratadas como string, pois representam categorias e não valores numéricos. Já variáveis indicadoras como WebPurchases e Complain devem ser convertidas para boolean, uma vez que assumem apenas dois estados e funcionam como marcadores de ocorrência.

Além disso, as variáveis relacionadas a gastos e compras, como os campos que começam com Mnt, foram ajustadas para float, considerando possíveis valores reais que fariam sentido nesses campos. Já as demais variáveis, que representam contagens ou valores discretos, foram mantidas como int, pois preservam a natureza original dos dados e evitam conversões desnecessárias.

In [67]:
base['WebPurchases'] = base['WebPurchases'].astype('boolean')
base['Complain'] = base['Complain'].astype('boolean')
base['Education'] = base['Education'].astype('string')
base['Marital_Status'] = base['Marital_Status'].astype('string')
base['MntWines'] = base['MntWines'].astype('float64')
base['MntFruits'] = base['MntFruits'].astype('float64')
base['MntMeatProducts'] = base['MntMeatProducts'].astype('float64')
base['MntFishProducts'] = base['MntFishProducts'].astype('float64')
base['MntSweetProducts'] = base['MntSweetProducts'].astype('float64')
base['MntGoldProds'] = base['MntGoldProds'].astype('float64')
base.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2240 entries, 0 to 2239
Data columns (total 16 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   Year_Birth         2240 non-null   int64  
 1   Education          2240 non-null   string 
 2   Marital_Status     2240 non-null   string 
 3   Income             2216 non-null   float64
 4   Kidhome            2240 non-null   int64  
 5   Recency            2240 non-null   int64  
 6   MntWines           2240 non-null   float64
 7   MntFruits          2240 non-null   float64
 8   MntMeatProducts    2240 non-null   float64
 9   MntFishProducts    2240 non-null   float64
 10  MntSweetProducts   2240 non-null   float64
 11  MntGoldProds       2240 non-null   float64
 12  NumStorePurchases  2240 non-null   int64  
 13  NumWebVisitsMonth  2240 non-null   int64  
 14  Complain           2240 non-null   boolean
 15  WebPurchases       2240 non-null   boolean
dtypes: boolean(2), float64(7

Após a correção dos tipos das variáveis, realizou-se uma verificação de integridade com foco em identificar registros inválidos e duplicidades no conjunto de dados. Nessa checagem, observou-se a presença de valores ausentes no campo de renda (Income), totalizando 24 ocorrências, além de 201 registros duplicados.

Para tratar essas inconsistências, adotou-se uma abordagem conservadora. No caso de Income, optou-se por inspecionar especificamente os registros com valores nulos ou NaN para avaliar se o problema estava restrito apenas ao campo de renda ou se os demais atributos também apresentavam incoerências. Essa análise é necessária para evitar a remoção indevida de registros potencialmente válidos, uma vez que a ausência de renda pode ocorrer por falha de preenchimento, enquanto o restante das informações pode continuar informativo para o modelo.

In [68]:
faltantes = base.isna().sum()
print(f"Dados faltantes: {faltantes}")

print(f"\nLinhas duplicadas: {base.duplicated().sum()}")

Dados faltantes: Year_Birth            0
Education             0
Marital_Status        0
Income               24
Kidhome               0
Recency               0
MntWines              0
MntFruits             0
MntMeatProducts       0
MntFishProducts       0
MntSweetProducts      0
MntGoldProds          0
NumStorePurchases     0
NumWebVisitsMonth     0
Complain              0
WebPurchases          0
dtype: int64

Linhas duplicadas: 201


In [69]:
base[base.isna().any(axis=1)]

,Year_Birth,Education,Marital_Status,Income,Kidhome,Recency,MntWines,MntFruits,MntMeatProducts,MntFishProducts,MntSweetProducts,MntGoldProds,NumStorePurchases,NumWebVisitsMonth,Complain,WebPurchases
10,1983,Graduation,Married,NaN,1,11,5.0,5.0,6.0,0.0,2.0,1.0,2,7,False,False
27,1986,Graduation,Single,NaN,1,19,5.0,1.0,3.0,3.0,263.0,362.0,0,1,False,True
43,1959,PhD,Single,NaN,0,80,81.0,11.0,50.0,3.0,2.0,39.0,4,2,False,False
48,1951,Graduation,Single,NaN,2,96,48.0,5.0,48.0,6.0,10.0,7.0,4,6,False,False
58,1982,Graduation,Single,NaN,1,57,11.0,3.0,22.0,2.0,2.0,6.0,3,6,False,False
71,1973,2n Cycle,Married,NaN,1,25,25.0,3.0,43.0,17.0,4.0,17.0,3,8,False,False
90,1957,PhD,Married,NaN,2,4,230.0,42.0,192.0,49.0,37.0,53.0,8,9,False,True
91,1957,Graduation,Single,NaN,1,45,7.0,0.0,8.0,2.0,0.0,1.0,2,7,False,False
92,1973,Master,Together,NaN,0,87,445.0,37.0,359.0,98.0,28.0,18.0,8,1,False,False
128,1961,PhD,Married,NaN,0,23,352.0,0.0,27.0,10.0,0.0,15.0,7,6,False,True


Ao inspecionar os registros que apresentavam valores inválidos em Income, observa-se que, fora esse campo, as demais variáveis permanecem preenchidas e coerentes com o padrão geral da base. Isso indica que não se tratam de linhas completamente inconsistentes, mas sim de um problema pontual de preenchimento da renda.

Dessa forma, a decisão tomada foi de preservar esses registros e substituir o valor ausente de Income pela mediana do grupo, nível educacional e estado civil, ao qual o cliente pertence, evitando perda de informação e mantendo a amostra o mais completa possível.

In [70]:
base['Income'] = base['Income'].fillna(base.groupby(['Education', 'Marital_Status'])['Income'].transform('median'))
base['Income'] = base['Income'].fillna(base['Income'].median())
print(base.isna().sum())

Year_Birth           0
Education            0
Marital_Status       0
Income               0
Kidhome              0
Recency              0
MntWines             0
MntFruits            0
MntMeatProducts      0
MntFishProducts      0
MntSweetProducts     0
MntGoldProds         0
NumStorePurchases    0
NumWebVisitsMonth    0
Complain             0
WebPurchases         0
dtype: int64


Em relação às duplicatas, procedeu-se à remoção dos registros repetidos, mantendo apenas a primeira ocorrência, de modo a preservar a versão original e evitar distorções estatísticas. A manutenção de duplicatas poderia inflar artificialmente padrões de comportamento e enviesar o treinamento, aumentando a chance do modelo aprender repetições em vez de relações reais entre variáveis e intenção de compra.

In [71]:
base = base.drop_duplicates()
print(f"\nLinhas duplicadas: {base.duplicated().sum()}")


Linhas duplicadas: 0


Como próximo passo da etapa de exploração dos dados, será realizada a análise visual do comportamento das variáveis. Inicialmente, as variáveis categóricas serão avaliadas por meio de histogramas, a fim de observar a distribuição das classes, identificar categorias predominantes e verificar possíveis desbalanceamentos que possam influenciar as interpretações e a modelagem.

Em seguida, as variáveis numéricas serão exploradas com o auxílio de boxplots, permitindo analisar a dispersão dos dados, a presença de assimetria e a ocorrência de outliers. Essa visualização é relevante para compreender o padrão de consumo e comportamento dos clientes, além de orientar decisões posteriores, como necessidade de transformações, padronização ou tratamento de valores extremos.

In [72]:
colunas_categorica = ['Education','Marital_Status','Kidhome', 'Complain','WebPurchases']
colunas_numericas = ['Year_Birth','Income','Recency','MntFruits','MntWines','MntMeatProducts','MntFishProducts','MntSweetProducts','MntGoldProds', 'NumStorePurchases','NumWebVisitsMonth']

cols = 3
rows = math.ceil(len(colunas_categorica)/cols)

hist = make_subplots(rows=rows, cols=cols, subplot_titles=colunas_categorica)

for i,col in enumerate(colunas_categorica):
  r = (i//cols) + 1
  c = (i%cols) + 1
  cont = base[col].value_counts()
  hist.add_trace(go.Bar(x=cont.index.tolist(), y=cont.values.tolist(), text=cont.values.tolist(), textposition="outside", name=col), row = r, col = c)

hist.update_layout(title="Distribuição das variáveis categóricas", showlegend=False, height=500*rows)
hist.show()

rows = math.ceil(len(colunas_numericas)/cols)

bplt = make_subplots(rows=rows, cols=cols, subplot_titles=colunas_numericas)
for i,col in enumerate(colunas_numericas):
  r = (i//cols)+1
  c = (i%cols)+1
  bplt.add_trace(go.Box(y=base[col], name=col, boxpoints='outliers'), row=r, col=c)

bplt.update_layout(title='Boxplot das variáveis numéricas', showlegend = False, height=500*rows)
bplt.show

<bound method BaseFigure.show of Figure({
    'data': [{'boxpoints': 'outliers',
              'name': 'Year_Birth',
              'type': 'box',
              'xaxis': 'x',
              'y': array([1957, 1954, 1965, ..., 1981, 1956, 1954]),
              'yaxis': 'y'},
             {'boxpoints': 'outliers',
              'name': 'Income',
              'type': 'box',
              'xaxis': 'x2',
              'y': array([58138., 46344., 71613., ..., 56981., 69245., 52869.]),
              'yaxis': 'y2'},
             {'boxpoints': 'outliers',
              'name': 'Recency',
              'type': 'box',
              'xaxis': 'x3',
              'y': array([58, 38, 26, ..., 91,  8, 40]),
              'yaxis': 'y3'},
             {'boxpoints': 'outliers',
              'name': 'MntFruits',
              'type': 'box',
              'xaxis': 'x4',
              'y': array([88.,  1., 49., ..., 48., 30.,  3.]),
              'yaxis': 'y4'},
             {'boxpoints': 'outliers',
              'name': 'MntWines',
              'type': 'box',
              'xaxis': 'x5',
              'y': array([635.,  11., 426., ..., 908., 428.,  84.]),
              'yaxis': 'y5'},
             {'boxpoints': 'outliers',
              'name': 'MntMeatProducts',
              'type': 'box',
              'xaxis': 'x6',
              'y': array([546.,   6., 127., ..., 217., 214.,  61.]),
              'yaxis': 'y6'},
             {'boxpoints': 'outliers',
              'name': 'MntFishProducts',
              'type': 'box',
              'xaxis': 'x7',
              'y': array([172.,   2., 111., ...,  32.,  80.,   2.]),
              'yaxis': 'y7'},
             {'boxpoints': 'outliers',
              'name': 'MntSweetProducts',
              'type': 'box',
              'xaxis': 'x8',
              'y': array([88.,  1., 21., ..., 12., 30.,  1.]),
              'yaxis': 'y8'},
             {'boxpoints': 'outliers',
              'name': 'MntGoldProds',
              'type': 'box',
              'xaxis': 'x9',
              'y': array([88.,  6., 42., ..., 24., 61., 21.]),
              'yaxis': 'y9'},
             {'boxpoints': 'outliers',
              'name': 'NumStorePurchases',
              'type': 'box',
              'xaxis': 'x10',
              'y': array([ 4,  2, 10, ..., 13, 10,  4]),
              'yaxis': 'y10'},
             {'boxpoints': 'outliers',
              'name': 'NumWebVisitsMonth',
              'type': 'box',
              'xaxis': 'x11',
              'y': array([7, 5, 4, ..., 6, 3, 7]),
              'yaxis': 'y11'}],
    'layout': {'annotations': [{'font': {'size': 16},
                                'showarrow': False,
                                'text': 'Year_Birth',
                                'x': 0.14444444444444446,
                                'xanchor': 'center',
                                'xref': 'paper',
                                'y': 1.0,
                                'yanchor': 'bottom',
                                'yref': 'paper'},
                               {'font': {'size': 16},
                                'showarrow': False,
                                'text': 'Income',
                                'x': 0.5,
                                'xanchor': 'center',
                                'xref': 'paper',
                                'y': 1.0,
                                'yanchor': 'bottom',
                                'yref': 'paper'},
                               {'font': {'size': 16},
                                'showarrow': False,
                                'text': 'Recency',
                                'x': 0.8555555555555556,
                                'xanchor': 'center',
                                'xref': 'paper',
                                'y': 1.0,
                                'yanchor': 'bottom',
                                'yref': 'paper'},
                               {'font': {'size': 16},
    

Ao analisar a base, observa-se a presença de valores inesperados e algumas inconsistências em variáveis que podem distorcer tanto a etapa de exploração quanto o desempenho do modelo. Dessa forma, optou-se por tratar esses pontos por meio do mapeamento correto das categorias, garantindo que os registros estejam coerentes e que cada classe seja representada de forma consistente no dataset.

In [73]:
#Marital_status:
map_status = {'Married':'Married', 'Together':'Together', 'Single': 'Single', 'Alone':'Single', "Divorced": "Divorced", "Widow": "Widow", "YOLO": "Unknown", "Absurd": "Unknown"}
base['Marital_Status'] = base['Marital_Status'].map(map_status)
base = base[base["Marital_Status"] != "Unknown"]
base["Marital_Status"].value_counts()

,count
Marital_Status,
Married,788
Together,517
Single,448
Divorced,213
Widow,70


In [74]:
rows = math.ceil(len(colunas_categorica)/cols)

hist = make_subplots(rows=rows, cols=cols, subplot_titles=colunas_categorica)

for i,col in enumerate(colunas_categorica):
  r = (i//cols) + 1
  c = (i%cols) + 1
  cont = base[col].value_counts()
  hist.add_trace(go.Bar(x=cont.index.tolist(), y=cont.values.tolist(), text=cont.values.tolist(), textposition="outside", name=col), row = r, col = c)

hist.update_layout(title="Distribuição das variáveis categóricas", showlegend=False, height=500*rows)
hist.show()

Além disso, identificou-se a presença de outliers que fogem do padrão esperado e não apresentam coerência com o contexto do problema. Esses valores extremos tendem a distorcer medidas estatísticas, influenciar visualizações e prejudicar o aprendizado do modelo, principalmente ao afetar médias, dispersões e relações entre variáveis.

Dessa forma, optou-se por remover esses registros outliers muito extremos antes da modelagem, com o objetivo de preservar a qualidade do conjunto de dados e garantir que o modelo aprenda padrões representativos do comportamento real dos clientes.

In [75]:
base = base[base['Year_Birth'] > 1940]

base = base[base['Income'] < 600000]

In [76]:
rows = math.ceil(len(colunas_numericas)/cols)

bplt = make_subplots(rows=rows, cols=cols, subplot_titles=colunas_numericas)
for i,col in enumerate(colunas_numericas):
  r = (i//cols)+1
  c = (i%cols)+1
  bplt.add_trace(go.Box(y=base[col], name=col, boxpoints='outliers'), row=r, col=c)

bplt.update_layout(title='Boxplot das variáveis numéricas', showlegend = False, height=500*rows)
bplt.show()

# ETAPA 2:
**Pré-processamento**

**Análise Correlação:** Verifique a correlação entre as váriaveis e análise se há espaço para retirar váriaveis que não te parecem importantes.

**Codificação de Variáveis Categóricas:** Transformar variáveis categóricas em um formato que os modelos de machine learning possam interpretar.


**Separe a base em Y, X e Treino e teste:**: Faça a separação da base.

**Realize a padronização dos dados**: Padronize os dados para garantir eficiência no modelo e eficácia.








Nessa segunda etapa, iniciou-se o pré processamento com a codificação das variáveis categóricas, permitindo que elas assumissem uma representação numérica interpretável por modelos de machine learning. Para isso, utilizou-se One Hot Encoding nas variáveis Marital_Status e Education, transformando cada categoria em uma coluna binária. Dessa forma, evita-se a criação de uma ordem artificial entre classes e preserva-se o significado categórico dessas variáveis para a etapa de modelagem.

In [77]:
categorias = ['Marital_Status', 'Education']

enc = OneHotEncoder(handle_unknown='ignore', sparse_output=False)
encoder = enc.fit_transform(base[categorias])

df = pd.DataFrame(encoder, columns=enc.get_feature_names_out(categorias), index=base.index).astype('int64')

base_encoded = pd.concat([base.drop(columns=categorias), df], axis=1)
base_encoded.info()

<class 'pandas.core.frame.DataFrame'>
Index: 2031 entries, 0 to 2239
Data columns (total 24 columns):
 #   Column                   Non-Null Count  Dtype  
---  ------                   --------------  -----  
 0   Year_Birth               2031 non-null   int64  
 1   Income                   2031 non-null   float64
 2   Kidhome                  2031 non-null   int64  
 3   Recency                  2031 non-null   int64  
 4   MntWines                 2031 non-null   float64
 5   MntFruits                2031 non-null   float64
 6   MntMeatProducts          2031 non-null   float64
 7   MntFishProducts          2031 non-null   float64
 8   MntSweetProducts         2031 non-null   float64
 9   MntGoldProds             2031 non-null   float64
 10  NumStorePurchases        2031 non-null   int64  
 11  NumWebVisitsMonth        2031 non-null   int64  
 12  Complain                 2031 non-null   boolean
 13  WebPurchases             2031 non-null   boolean
 14  Marital_Status_Divorced  2031

Em seguida, decidiu-se utilizar a matriz de correlação para analisar o grau de associação entre as variáveis numéricas e identificar quais atributos apresentam maior relação com o comportamento de compra no site. Essa análise auxilia a reconhecer padrões relevantes, detectar possíveis redundâncias entre variáveis e orientar decisões posteriores de seleção ou criação de atributos.

In [78]:
corr = base_encoded.corr(numeric_only=True)
fig = px.imshow(corr, text_auto='.2f', aspect='auto', title='Matriz de Correlação', color_continuous_scale='Greens')
fig.update_layout(width=950, height=800, xaxis_title='', yaxis_title='')
fig.show()

Olhando para a matriz de correlação, observa-se que a variável WebPurchases apresenta associação mais forte com indicadores de histórico de consumo e perfil de compra do cliente. Em especial, destacam-se a relação com o número de compras na loja física (NumStorePurchases), com os valores gastos nas categorias de produtos (como vinho, produtos gold, carne, doces e frutas) e também com a variável Kidhome, que representa a quantidade de crianças na residência.

Em seguida, como última parte da etapa de pré processamento, a base foi separada em variáveis explicativas (X) e variável alvo (y), preparando o conjunto de dados para o treinamento do modelo. Além disso, definiu-se a estratégia de validação cruzada (cv).

In [79]:
x = base_encoded.drop(columns=["WebPurchases"])
y = base_encoded["WebPurchases"].astype(int)

cv = KFold(n_splits=5, shuffle=True, random_state=42)

# ETAPA 3:

**Modelagem**

Escolha ao menos 2 técnicas de machine learning e rode 2 modelos, afim de identificar qual tem o melhor resultado para essa base. Lembrando que estamos lidando com uma classificação binária.

Para a etapa de modelagem, serão utilizados dois algoritmos distintos, Random Forest e Regressão Logística, permitindo comparar um método não linear baseado em árvores com um modelo linear mais interpretável. Como a Regressão Logística é sensível à escala das variáveis, foi necessário realizar a padronização dos dados, garantindo que atributos com magnitudes diferentes não dominem o processo de ajuste do modelo. Dessa forma, as variáveis numéricas foram transformadas para uma escala comparável antes do treinamento, favorecendo estabilidade e melhor desempenho na regressão.

In [80]:
logistic_regression = Pipeline([("scaler", StandardScaler()), ("clf", LogisticRegression(max_iter=3000))])
scores_lr = cross_val_score(logistic_regression, x, y, cv=cv, scoring="f1")
print("LogisticRegression f1:", float(scores_lr.mean()), float(scores_lr.std()))

random_forest = RandomForestClassifier(random_state=42)
scores_rf = cross_val_score(random_forest, x, y, cv=cv, scoring="f1")
print("RandomForest f1:", float(scores_rf.mean()), float(scores_rf.std()))

LogisticRegression f1: 0.8365394986181485 0.009126167301985337
RandomForest f1: 0.8996368895103071 0.011692760554723092


Ao comparar os modelos utilizando F1 score com validação cruzada, observa-se que a Random Forest apresentou melhor resultado, com F1 médio de 0,900 e desvio padrão de 0,012, enquanto a Regressão Logística obteve 0,837 com desvio padrão de 0,009. Esses valores indicam que a Random Forest alcançou um equilíbrio superior entre precision e recall na identificação de compras no site, mantendo desempenho consistente entre os folds da validação.

Entretanto, até esse momento, o treinamento foi realizado utilizando o conjunto completo de variáveis disponíveis e apesar do uso de validação cruzada fornecer uma estimativa mais robusta de generalização, a base apresenta alta dimensionalidade após a codificação das variáveis categóricas, o que pode introduzir redundância e aumentar a complexidade do espaço de atributos. Dessa forma, avaliou-se a aplicação de PCA para o algoritmo de regressão logística como técnica de redução de dimensionalidade para condensar a informação em menos componentes e investigar possíveis ganhos em simplicidade e estabilidade do modelo, especialmente para a Regressão Logística.

In [81]:
logistic_pca = Pipeline([("scaler", StandardScaler()), ("pca", PCA(n_components=0.95, random_state=42)), ("clf", LogisticRegression(max_iter=3000))])
scores_lr_pca = cross_val_score(logistic_pca, x, y, cv=cv, scoring="f1")
print("LogisticRegression PCA f1:", float(scores_lr_pca.mean()), float(scores_lr_pca.std()))

LogisticRegression PCA f1: 0.8297783642467816 0.016732059816696915


A aplicação de PCA foi testada como estratégia de redução de dimensionalidade após a padronização das variáveis. Entretanto, ao comparar os resultados com validação cruzada, observou-se uma leve redução no desempenho da Regressão Logística, com F1 médio passando de aproximadamente 0,837 para 0,830, além de aumento no desvio padrão entre os folds. Dessa forma, conclui-se que, para este conjunto de dados, a redução via PCA não trouxe melhoria de desempenho e pode ter removido parte da informação discriminativa relevante, sendo mantida a versão sem PCA para a Regressão Logística.

# ETAPA 4:

**Avaliação**

Avalie os resultados encontrados nos dois modelos e identifique qual te pareceu realizar melhor as previsões.

Utilize além das métricas padrões a matriz de confusão.

In [85]:
y_pred_lr = cross_val_predict(logistic_regression, x, y, cv=cv, method="predict")
cm_lr = confusion_matrix(y, y_pred_lr)
print("Regressão Logística:")
print(classification_report(y, y_pred_lr))

fig = px.imshow(cm_lr, text_auto=True, aspect="auto", title="Confusion matrix | LogisticRegression", color_continuous_scale="Greens", labels=dict(x="Predicted", y="Actual"))
fig.update_xaxes(tickvals=[0, 1], ticktext=["0", "1"])
fig.update_yaxes(tickvals=[0, 1], ticktext=["0", "1"])
fig.show()

Regressão Logística:
              precision    recall  f1-score   support

           0       0.83      0.84      0.83      1004
           1       0.84      0.83      0.84      1027

    accuracy                           0.84      2031
   macro avg       0.84      0.84      0.84      2031
weighted avg       0.84      0.84      0.84      2031



Ao avaliar os resultados obtidos com a Regressão Logística, é possível perceber que o modelo apresentou desempenho equilibrado entre as classes compra e não compra, com acurácia global de 84%, indicando que aproximadamente 84% das observações do conjunto de teste foram corretamente classificadas. As métricas por classe permaneceram muito próximas, com precision e recall em torno de 83% a 84% para ambas as classes. Esse comportamento sugere que o modelo não favorece excessivamente um grupo em detrimento do outro, mantendo um trade off estável entre identificar compradores e evitar previsões incorretas de compra. Ainda assim, observa-se a presença de falsos negativos em volume significativo, o que representa casos em que clientes que efetivamente comprariam não seriam identificados como prioridade para ações de conversão.

In [86]:
y_pred_rf = cross_val_predict(random_forest, x, y, cv=cv, method="predict")
cm_rf = confusion_matrix(y, y_pred_rf)
print("Random Forest:")
print(classification_report(y, y_pred_rf))

fig = px.imshow(cm_rf, text_auto=True, aspect="auto", title="Confusion matrix | RandomForest", color_continuous_scale="Greens", labels=dict(x="Predicted", y="Actual"))
fig.update_xaxes(tickvals=[0, 1], ticktext=["0", "1"])
fig.update_yaxes(tickvals=[0, 1], ticktext=["0", "1"])
fig.show()

Random Forest:
              precision    recall  f1-score   support

           0       0.93      0.85      0.89      1004
           1       0.87      0.94      0.90      1027

    accuracy                           0.89      2031
   macro avg       0.90      0.89      0.89      2031
weighted avg       0.90      0.89      0.89      2031



Para a Random Forest, obtivemos um desempenho superior, com acurácia global de 89%, 5% acima da Regressão Logística. Além do ganho em acurácia, o principal destaque encontra-se na classe de compra, em que o modelo alcançou recall de 94%, o que indica alta capacidade de capturar compradores reais, reduzindo a ocorrência de falsos negativos. Essa característica torna a random forest bastante atrativa, pois diminui a probabilidade de deixar de direcionar ações a clientes com potencial real de conversão. Em relação aos falsos positivos, observa-se que o modelo também apresentou um resultado ligeiramente melhor do que o obtido pela Regressão Logística, reduzindo esse tipo de erro, ainda que ele permaneça presente. Dessa forma, apesar de ainda existirem casos em que o modelo prevê compra e o cliente não compra, o volume de acionamentos desnecessários tende a ser mais controlado, mantendo um trade off favorável entre capturar conversões e evitar abordagens indevidas.

Dessa forma, embora ambos os modelos apresentem desempenho adequado, a Random Forest se mostra mais indicada para o objetivo do projeto, pois combina melhor desempenho global com maior capacidade de identificar corretamente clientes que comprariam, reduzindo perdas potenciais de conversão.